# SEPA Precios — Exploración de Productos y Canasta Representativa

**Objetivo:** Explorar los datos del SEPA de **abril 2026** para identificar qué productos tienen alta cobertura a nivel nacional (cadenas y regiones) y construir una **canasta representativa para una familia tipo de 4 integrantes**.

**Fuentes de datos:**
- [SEPA — Secretaría de Comercio Argentina](https://datos.produccion.gob.ar/dataset/sepa-precios): precios diarios reportados por cadenas de supermercados
- `Maestro de Productos Interno.xlsx`: clasificación de productos (rubro, categoría, subcategoría)
- `maestro_sucursales_completo.xlsx`: metadata de sucursales (cadena, región geográfica)

**Estructura del notebook:**
1. Configuración y carga de datos
2. Exploración inicial (cadenas, regiones, productos)
3. Análisis de cobertura por cadena y región
4. Construcción y validación de la canasta
5. Exportación de resultados

---
> **Nota sobre el formato de precios:** Los precios en la base SEPA están almacenados como enteros en centavos (ej: `1699999` = `$16.999,99`). El notebook los divide por 100 para trabajar en pesos. Verificar con la celda de validación de precios (Sección 2).

## 1. Configuración del entorno

In [ ]:
# Instalar dependencias no disponibles por defecto en Colab
!pip install openpyxl -q

import zipfile
import gzip
import io
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Librerías cargadas correctamente')

In [ ]:
# ===========================================================
# CONFIGURACIÓN — Ajustar rutas según ubicación en Google Drive
# ===========================================================

# Ruta al ZIP con los datos SEPA de 2026 (primer semestre)
# El ZIP contiene archivos .csv.gz con precios diarios por mes
SEPA_ZIP_PATH = '/content/drive/MyDrive/SEPA/2026A.zip'

# Archivos de abril 2026 dentro del ZIP
# Parte 1: días 1 al 15 | Parte 2: días 16 al 30
ABRIL_PARTE1 = '042026_pais_parte1COMPLETO.csv.gz'
ABRIL_PARTE2 = '042026_pais_parte2COMPLETO.csv.gz'

# Maestros (subir a Drive y ajustar rutas)
MAESTRO_PRODUCTOS_PATH = '/content/drive/MyDrive/SEPA/Maestro de Productos Interno.xlsx'
MAESTRO_SUCURSALES_PATH = '/content/drive/MyDrive/SEPA/maestro_sucursales_completo.xlsx'

# Directorio de salida para la canasta y gráficos
OUTPUT_DIR = '/content/drive/MyDrive/SEPA/output_canasta_abril2026'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===========================================================
# PARÁMETROS DE COBERTURA PARA LA CANASTA (ajustables)
# ===========================================================
# El SEPA de abril 2026 reporta 6 cadenas y 6 regiones.
# Los umbrales definen qué tan representativo debe ser un producto.

MIN_CADENAS   = 3     # Presente en al menos 3 de las 6 cadenas (>=50%)
MIN_REGIONES  = 4     # Presente en al menos 4 de las 6 regiones (>=67%)
MIN_SUCURSALES = 50   # Reportado por al menos 50 sucursales distintas
MIN_PCT_DIAS  = 0.50  # Precio disponible al menos el 50% de los días de abril

print('Configuración cargada')
print(f'  ZIP: {SEPA_ZIP_PATH}')
print(f'  Umbrales: {MIN_CADENAS} cadenas | {MIN_REGIONES} regiones | {MIN_SUCURSALES} sucursales | {MIN_PCT_DIAS*100:.0f}% días')

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive montado correctamente')

## 2. Carga de datos SEPA — Abril 2026

Los archivos SEPA siguen el formato: `MMAAAA_pais_parteNCompleto.csv.gz` dentro de un ZIP semestral.
- **Columnas clave:** `id_comercio`, `id_bandera`, `id_sucursal`, `sucursales_provincia`, `id_producto`
- **Columnas de precio:** una por día del mes (`precio_AAAAMMDD`), valor entero en centavos, `NA` si no se reportó ese día

In [ ]:
def cargar_sepa(zip_path: str, filename: str) -> pd.DataFrame:
    """
    Lee un archivo .csv.gz desde dentro de un .zip.
    Devuelve un DataFrame reducido: identificadores + precio_promedio + dias_con_precio.
    Los precios se convierten de centavos a pesos (/ 100).
    """
    print(f'  Leyendo {filename} ...')

    # Leer el .gz comprimido desde dentro del .zip al buffer en memoria
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(filename) as zf:
            buf = io.BytesIO(zf.read())

    with gzip.open(buf, 'rt', encoding='utf-8') as g:
        df = pd.read_csv(
            g,
            dtype={
                'id_comercio': 'str',
                'id_bandera': 'str',
                'id_sucursal': 'str',
                'id_producto': 'str',
                'sucursales_provincia': 'str'
            },
            low_memory=False
        )

    price_cols = [c for c in df.columns if c.startswith('precio_')]
    n_dias = len(price_cols)

    # Convertir a numérico y transformar centavos → pesos
    df[price_cols] = df[price_cols].replace('NA', np.nan).astype(float) / 100

    # Calcular métricas por fila (producto × sucursal)
    df['precio_promedio']  = df[price_cols].mean(axis=1)
    df['dias_con_precio']  = df[price_cols].notna().sum(axis=1)
    df['total_dias_parte'] = n_dias

    filas = len(df)
    productos = df['id_producto'].nunique()
    print(f'    -> {filas:,} filas | {productos:,} productos únicos | {n_dias} días')

    # Reducir columnas para ahorrar memoria
    return df[['id_comercio', 'id_bandera', 'id_sucursal', 'sucursales_provincia',
               'id_producto', 'precio_promedio', 'dias_con_precio', 'total_dias_parte']]

In [ ]:
import gc

print('Cargando datos SEPA de Abril 2026...')
print('-' * 55)

df_p1 = cargar_sepa(SEPA_ZIP_PATH, ABRIL_PARTE1)
df_p2 = cargar_sepa(SEPA_ZIP_PATH, ABRIL_PARTE2)

# Concatenar las dos partes del mes
df_abril = pd.concat([df_p1, df_p2], ignore_index=True)
del df_p1, df_p2
gc.collect()

# Consolidar por (producto × sucursal): sumar días y promediar precio
df_suc = df_abril.groupby(
    ['id_producto', 'id_bandera', 'id_comercio', 'id_sucursal', 'sucursales_provincia'],
    as_index=False
).agg(
    precio_promedio   = ('precio_promedio',   'mean'),
    dias_con_precio   = ('dias_con_precio',   'sum'),
    total_dias        = ('total_dias_parte',  'sum')
)
df_suc['pct_dias'] = df_suc['dias_con_precio'] / df_suc['total_dias']

del df_abril
gc.collect()

print()
print('Datos consolidados (producto x sucursal):')
print(f'  Filas:              {len(df_suc):,}')
print(f'  Productos únicos:   {df_suc["id_producto"].nunique():,}')
print(f'  Cadenas (bandera):  {df_suc["id_bandera"].nunique()}')
print(f'  Provincias:         {df_suc["sucursales_provincia"].nunique()}')
print(f'  Sucursales:         {df_suc["id_sucursal"].nunique():,}')

In [ ]:
# ------------------------------------------------------------------
# VERIFICACIÓN DE ESCALA DE PRECIOS
# Muestra los 10 productos con más observaciones y sus precios.
# Confirmar que la conversión /100 (centavos → pesos) es correcta.
# ------------------------------------------------------------------
print('=== Verificación de escala de precios ===')
print('Los precios se convirtieron dividiendo por 100 (centavos → pesos)')
print()

top_obs = (
    df_suc.groupby('id_producto')
    .agg(n_sucursales=('id_sucursal','count'),
         precio_mediano=('precio_promedio','median'))
    .sort_values('n_sucursales', ascending=False)
    .head(10)
    .reset_index()
)
print(top_obs.to_string(index=False))
print()
print('Si los precios son muy altos o muy bajos, ajustar el divisor en cargar_sepa().')
print('Divisores comunes: /100 (centavos→pesos) o /1000 (milipesos→pesos)')

## 3. Carga de maestros

In [ ]:
# ---- Maestro de Productos ----
print('Cargando Maestro de Productos...')

df_prod = pd.read_excel(
    MAESTRO_PRODUCTOS_PATH,
    dtype={'producto_sepa_id': str}
)

# Normalizar id y excluir productos en blacklist
df_prod['id_producto'] = df_prod['producto_sepa_id'].str.strip()
df_prod = df_prod[df_prod['producto_blacklist'] == 0].copy()

# Quedarse con una fila por id_producto (puede haber duplicados entre comercios)
df_prod_uniq = (
    df_prod[['id_producto', 'producto_descripcion', 'producto_marca',
             'rubro', 'categoria', 'subcategoria',
             'producto_cantidad_presentacion', 'producto_unidad_medida_presentac']]
    .drop_duplicates('id_producto')
)

print(f'  Productos únicos (sin blacklist): {len(df_prod_uniq):,}')
print(f'  Rubros disponibles ({df_prod_uniq["rubro"].nunique()}):')
print(df_prod_uniq['rubro'].value_counts().to_string())

In [ ]:
# ---- Maestro de Sucursales ----
print('Cargando Maestro de Sucursales...')

df_suc_maest = pd.read_excel(
    MAESTRO_SUCURSALES_PATH,
    dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}
)

# Limpiar nombres de región (pueden tener espacios)
df_suc_maest['REGION'] = df_suc_maest['REGION'].str.strip()

print(f'  Total sucursales: {len(df_suc_maest):,}')
print(f'  Cadenas (id_bandera): {df_suc_maest["id_bandera"].nunique()}')
print(f'  Regiones ({df_suc_maest["REGION"].nunique()}):')
print(df_suc_maest.groupby('REGION')['id_sucursal'].nunique()
      .sort_values(ascending=False).to_string())

print('\nCadenas por sucursales:')
print(df_suc_maest.groupby('id_bandera')['id_sucursal'].nunique()
      .sort_values(ascending=False).to_string())

## 4. Enriquecimiento y exploración

In [ ]:
# Unir datos SEPA con maestro de sucursales para obtener REGION y nombre de cadena
suc_info = df_suc_maest[['id_comercio', 'id_bandera', 'id_sucursal',
                          'sucursales_nombre', 'PROVINCIA', 'REGION']].copy()

df_enr = df_suc.merge(suc_info, on=['id_comercio', 'id_bandera', 'id_sucursal'], how='left')
df_enr['REGION'] = df_enr['REGION'].str.strip()

# Unir con maestro de productos
df_enr = df_enr.merge(df_prod_uniq, on='id_producto', how='left')

pct_suc  = df_enr['REGION'].notna().mean() * 100
pct_prod = df_enr['rubro'].notna().mean() * 100
print(f'Match con maestro sucursales: {pct_suc:.1f}% de filas')
print(f'Match con maestro productos:  {pct_prod:.1f}% de filas')
print(f'Productos sin clasificar en maestro: {df_enr[df_enr["rubro"].isna()]["id_producto"].nunique():,}')

In [ ]:
# ---- Exploración: distribución por cadena ----
print('=== Distribución por cadena (id_bandera) — Abril 2026 ===')
resumen_cadena = df_enr.groupby('id_bandera').agg(
    sucursales_activas = ('id_sucursal', 'nunique'),
    productos_reportados = ('id_producto', 'nunique'),
    precio_mediano = ('precio_promedio', 'median')
).sort_values('productos_reportados', ascending=False)
print(resumen_cadena.to_string())

print('\n=== Distribución por región geográfica ===')
resumen_region = df_enr.groupby('REGION').agg(
    sucursales_activas = ('id_sucursal', 'nunique'),
    productos_reportados = ('id_producto', 'nunique')
).sort_values('sucursales_activas', ascending=False)
print(resumen_region.to_string())

In [ ]:
# ---- Exploración: productos más reportados (sin maestro) ----
print('=== Top 20 productos por sucursales que los reportan ===')
top_prod = (
    df_enr.groupby(['id_producto', 'producto_descripcion', 'producto_marca', 'rubro', 'categoria'])
    .agg(
        n_sucursales = ('id_sucursal', 'count'),
        n_cadenas    = ('id_bandera', 'nunique'),
        n_regiones   = ('REGION', lambda x: x.dropna().nunique()),
        precio_med   = ('precio_promedio', 'median')
    )
    .sort_values('n_sucursales', ascending=False)
    .head(20)
    .reset_index()
)
print(top_prod[['id_producto','producto_descripcion','producto_marca',
                'rubro','n_cadenas','n_regiones','n_sucursales','precio_med']].to_string(index=False))

## 5. Análisis de cobertura de productos

Para cada producto se calculan:
- **n_cadenas**: cadenas distintas que lo reportaron en abril
- **n_regiones**: regiones geográficas con al menos una sucursal que lo reportó
- **n_sucursales**: total de sucursales que lo reportaron
- **pct_dias_promedio**: porcentaje promedio de días del mes con precio disponible
- **score_cobertura**: índice compuesto = 50% cobertura cadenas + 50% cobertura regiones, ponderado por continuidad de reporte

In [ ]:
total_cadenas  = df_enr['id_bandera'].nunique()
total_regiones = df_enr['REGION'].dropna().nunique()

print(f'Cadenas activas en abril 2026:  {total_cadenas}')
print(f'Regiones activas en abril 2026: {total_regiones}')

# Calcular cobertura por producto
df_cob = df_enr.groupby('id_producto').agg(
    n_cadenas         = ('id_bandera',       'nunique'),
    n_regiones        = ('REGION',           lambda x: x.dropna().nunique()),
    n_sucursales      = ('id_sucursal',      'count'),
    pct_dias_promedio = ('pct_dias',         'mean'),
    precio_mediano    = ('precio_promedio',  'median'),
    precio_promedio   = ('precio_promedio',  'mean'),
    precio_p25        = ('precio_promedio',  lambda x: x.quantile(0.25)),
    precio_p75        = ('precio_promedio',  lambda x: x.quantile(0.75)),
    rubro             = ('rubro',            'first'),
    categoria         = ('categoria',        'first'),
    subcategoria      = ('subcategoria',     'first'),
    descripcion       = ('producto_descripcion', 'first'),
    marca             = ('producto_marca',   'first'),
    presentacion      = ('producto_cantidad_presentacion', 'first'),
    unidad            = ('producto_unidad_medida_presentac', 'first')
).reset_index()

df_cob['pct_cadenas']   = df_cob['n_cadenas']  / total_cadenas
df_cob['pct_regiones']  = df_cob['n_regiones'] / total_regiones
df_cob['score_cobertura'] = (
    (df_cob['pct_cadenas'] * 0.5 + df_cob['pct_regiones'] * 0.5)
    * df_cob['pct_dias_promedio']
)

print(f'\nProductos con al menos 1 observación: {len(df_cob):,}')
print(f'\nDistribución de cobertura:')
print(df_cob[['n_cadenas','n_regiones','n_sucursales','pct_dias_promedio']].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribución de Cobertura — SEPA Abril 2026', fontsize=14, fontweight='bold')

# Cadenas
ax = axes[0, 0]
ax.hist(df_cob['n_cadenas'], bins=range(0, total_cadenas + 2), color='steelblue', edgecolor='white', align='left')
ax.axvline(MIN_CADENAS - 0.5, color='crimson', linestyle='--', linewidth=1.5, label=f'Umbral: {MIN_CADENAS}')
ax.set_title('N° de cadenas que reportan el producto')
ax.set_xlabel('Cadenas')
ax.set_ylabel('N° de productos')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend()

# Regiones
ax = axes[0, 1]
ax.hist(df_cob['n_regiones'], bins=range(0, total_regiones + 2), color='seagreen', edgecolor='white', align='left')
ax.axvline(MIN_REGIONES - 0.5, color='crimson', linestyle='--', linewidth=1.5, label=f'Umbral: {MIN_REGIONES}')
ax.set_title('N° de regiones con cobertura')
ax.set_xlabel('Regiones')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend()

# Sucursales (cap. 500 para legibilidad)
ax = axes[1, 0]
ax.hist(df_cob['n_sucursales'].clip(upper=600), bins=40, color='darkorange', edgecolor='white')
ax.axvline(MIN_SUCURSALES, color='crimson', linestyle='--', linewidth=1.5, label=f'Umbral: {MIN_SUCURSALES}')
ax.set_title('N° de sucursales (cap. 600)')
ax.set_xlabel('Sucursales')
ax.legend()

# % días reportados
ax = axes[1, 1]
ax.hist(df_cob['pct_dias_promedio'], bins=25, color='mediumpurple', edgecolor='white')
ax.axvline(MIN_PCT_DIAS, color='crimson', linestyle='--', linewidth=1.5, label=f'Umbral: {MIN_PCT_DIAS*100:.0f}%')
ax.set_title('% días del mes con precio reportado')
ax.set_xlabel('% días')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_distribucion_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()

# Cuántos productos superan cada umbral
n_cad = (df_cob['n_cadenas']  >= MIN_CADENAS).sum()
n_reg = (df_cob['n_regiones'] >= MIN_REGIONES).sum()
n_suc = (df_cob['n_sucursales'] >= MIN_SUCURSALES).sum()
n_dias = (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS).sum()
n_todos = (
    (df_cob['n_cadenas']  >= MIN_CADENAS) &
    (df_cob['n_regiones'] >= MIN_REGIONES) &
    (df_cob['n_sucursales'] >= MIN_SUCURSALES) &
    (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS)
).sum()
print(f'Productos que superan umbral de cadenas:    {n_cad:,} / {len(df_cob):,}')
print(f'Productos que superan umbral de regiones:   {n_reg:,} / {len(df_cob):,}')
print(f'Productos que superan umbral de sucursales: {n_suc:,} / {len(df_cob):,}')
print(f'Productos que superan umbral de días:       {n_dias:,} / {len(df_cob):,}')
print(f'--------------------------------------------------')
print(f'Productos que superan TODOS los umbrales:   {n_todos:,} / {len(df_cob):,}')

In [ ]:
# Aplicar filtros para obtener productos candidatos a la canasta
mask = (
    (df_cob['n_cadenas']          >= MIN_CADENAS)   &
    (df_cob['n_regiones']         >= MIN_REGIONES)  &
    (df_cob['n_sucursales']       >= MIN_SUCURSALES) &
    (df_cob['pct_dias_promedio']  >= MIN_PCT_DIAS)
)
candidatos = df_cob[mask].copy()
cand_con_maestro = candidatos[candidatos['rubro'].notna()].copy()

print(f'Productos candidatos (todos los filtros): {len(candidatos):,}')
print(f'  Con clasificación en maestro:           {len(cand_con_maestro):,}')
print(f'  Sin clasificar (id no está en maestro): {len(candidatos) - len(cand_con_maestro):,}')
print('\nCandidatos por rubro:')
print(cand_con_maestro['rubro'].value_counts().to_string())

In [ ]:
# Heatmap: productos top × cadenas — cobertura geográfica
top_ids = (
    cand_con_maestro.sort_values('score_cobertura', ascending=False)
    .head(40)['id_producto'].tolist()
)

df_heat_data = df_enr[
    df_enr['id_producto'].isin(top_ids)
].merge(
    candidatos[['id_producto', 'descripcion']], on='id_producto', how='left'
)
df_heat_data['label'] = df_heat_data['descripcion'].str[:45].fillna(df_heat_data['id_producto'])

pivot_heat = (
    df_heat_data.groupby(['label', 'id_bandera'])['pct_dias']
    .mean()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(
    pivot_heat,
    cmap='YlGnBu', linewidths=0.4, linecolor='white',
    vmin=0, vmax=1,
    cbar_kws={'label': '% días con precio', 'shrink': 0.6},
    ax=ax
)
ax.set_title('Cobertura por producto × cadena — Top 40 candidatos', fontsize=13, pad=12)
ax.set_xlabel('Cadena (id_bandera)', fontsize=11)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_heatmap_cobertura_cadenas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: productos top × regiones
pivot_reg = (
    df_heat_data.dropna(subset=['REGION'])
    .groupby(['label', 'REGION'])['pct_dias']
    .mean()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(
    pivot_reg,
    cmap='RdYlGn', linewidths=0.4, linecolor='white',
    vmin=0, vmax=1,
    cbar_kws={'label': '% días con precio', 'shrink': 0.6},
    ax=ax
)
ax.set_title('Cobertura por producto × región — Top 40 candidatos', fontsize=13, pad=12)
ax.set_xlabel('Región', fontsize=11)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_heatmap_cobertura_regiones.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Construcción de la canasta representativa

La canasta se arma siguiendo la estructura de la **Canasta Básica Alimentaria (CBA)** del INDEC para una familia tipo de 4 integrantes, adaptada a los rubros y categorías del sistema SEPA.

**Criterio de selección:** dentro de cada grupo, se seleccionan los productos con mayor `score_cobertura` (que combina presencia en cadenas, regiones y continuidad de reporte).

In [ ]:
# ===========================================================
# DEFINICIÓN DE GRUPOS DE LA CANASTA
# Cada grupo define: rubros SEPA elegibles, palabras clave
# para filtrar categorías, y cantidad máxima de productos.
# Ajustar según los rubros/categorías que aparecen en el maestro.
# ===========================================================

GRUPOS_CANASTA = {
    'Cereales y derivados': {
        'rubros': ['Almacén'],
        'keywords_cat': ['arroz', 'pasta', 'fideo', 'harina', 'galletita', 'cereal', 'pan'],
        'max_productos': 8
    },
    'Lácteos': {
        'rubros': ['Frescos', 'Almacén'],
        'keywords_cat': ['leche', 'yogur', 'queso', 'crema', 'postre'],
        'max_productos': 8
    },
    'Aceites y grasas': {
        'rubros': ['Almacén'],
        'keywords_cat': ['aceite', 'manteca', 'margarina'],
        'max_productos': 4
    },
    'Azúcar, dulces y conservas': {
        'rubros': ['Almacén'],
        'keywords_cat': ['azúcar', 'azucar', 'mermelada', 'dulce', 'tomate', 'conserva', 'legumbre'],
        'max_productos': 6
    },
    'Carnes y fiambres': {
        'rubros': ['Frescos', 'Almacén', 'Congelados'],
        'keywords_cat': ['fiambre', 'embutido', 'carne', 'salchicha', 'pollo', 'atún', 'atun'],
        'max_productos': 6
    },
    'Huevos': {
        'rubros': ['Frescos', 'Almacén'],
        'keywords_cat': ['huevo'],
        'max_productos': 2
    },
    'Condimentos y aderezos': {
        'rubros': ['Almacén'],
        'keywords_cat': ['salsa', 'condimento', 'vinagre', 'mayonesa', 'mostaza', 'ketchup', 'aderezo'],
        'max_productos': 5
    },
    'Bebidas no alcohólicas': {
        'rubros': ['Bebidas'],
        'keywords_cat': ['agua', 'gaseosa', 'jugo', 'saborizada', 'infusión', 'infusion', 'te', 'café', 'cafe'],
        'max_productos': 7
    },
    'Bebidas alcohólicas': {
        'rubros': ['Bebidas'],
        'keywords_cat': ['cerveza', 'vino', 'sidra', 'fernet', 'espirituosa'],
        'max_productos': 4
    },
    'Limpieza del hogar': {
        'rubros': ['Limpieza'],
        'keywords_cat': None,  # Todo el rubro Limpieza
        'max_productos': 7
    },
    'Higiene y cuidado personal': {
        'rubros': ['Perfumería'],
        'keywords_cat': None,  # Todo el rubro Perfumería
        'max_productos': 6
    },
}

print(f'Grupos definidos para la canasta: {len(GRUPOS_CANASTA)}')
for g, v in GRUPOS_CANASTA.items():
    print(f'  {g}: máx {v["max_productos"]} productos | rubros: {v["rubros"]}')

In [ ]:
def seleccionar_grupo(candidatos_df, rubros, keywords_cat, max_n):
    """
    Filtra candidatos por rubro y palabras clave en categoría.
    Si no hay suficientes productos con keyword, amplía al rubro completo.
    Devuelve top max_n por score_cobertura.
    """
    subset = candidatos_df[candidatos_df['rubro'].isin(rubros)].copy()

    if keywords_cat and len(subset) > 0:
        pattern = '|'.join(keywords_cat)
        mask_kw = subset['categoria'].str.contains(pattern, case=False, na=False)
        subset_kw = subset[mask_kw]
        # Usar keywords si hay suficientes; si no, caer al rubro completo
        subset = subset_kw if len(subset_kw) >= 2 else subset

    return subset.sort_values('score_cobertura', ascending=False).head(max_n)


# Construir la canasta
partes = []
for grupo, cfg in GRUPOS_CANASTA.items():
    seleccion = seleccionar_grupo(
        cand_con_maestro,
        cfg['rubros'],
        cfg['keywords_cat'],
        cfg['max_productos']
    )
    seleccion = seleccion.copy()
    seleccion['grupo_canasta'] = grupo
    partes.append(seleccion)
    print(f'{grupo}: {len(seleccion)} productos seleccionados')

df_canasta = pd.concat(partes, ignore_index=True)
# Eliminar duplicados que podrían haber caído en más de un grupo
df_canasta = df_canasta.drop_duplicates(subset='id_producto', keep='first')

print(f'\nTotal productos en la canasta: {len(df_canasta)}')

In [ ]:
# Mostrar la canasta completa por grupo
cols_show = ['descripcion', 'marca', 'presentacion', 'unidad',
             'n_cadenas', 'n_regiones', 'n_sucursales',
             'pct_dias_promedio', 'precio_mediano']

print('=' * 110)
print('CANASTA REPRESENTATIVA — FAMILIA TIPO 4 INTEGRANTES — ABRIL 2026')
print('=' * 110)

for grupo in GRUPOS_CANASTA.keys():
    gdf = df_canasta[df_canasta['grupo_canasta'] == grupo]
    if len(gdf) == 0:
        print(f'\n[{grupo}] — sin productos que cumplan los umbrales')
        continue
    print(f'\n{"─" * 110}')
    print(f'  {grupo.upper()}  ({len(gdf)} productos)')
    print(f'{"─" * 110}')
    print(
        gdf[cols_show]
        .sort_values('n_cadenas', ascending=False)
        .to_string(index=False)
    )

In [ ]:
# ---- Gráfico resumen de la canasta ----
resumen_canasta = (
    df_canasta.groupby('grupo_canasta')
    .agg(
        n_productos    = ('id_producto',      'count'),
        cob_cadenas    = ('n_cadenas',         'mean'),
        cob_regiones   = ('n_regiones',        'mean'),
        precio_mediano = ('precio_mediano',    'median')
    )
    .reset_index()
    .sort_values('cob_cadenas', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Resumen de la Canasta Representativa — Abril 2026', fontsize=13, fontweight='bold')

# Cobertura promedio por grupo
ax = axes[0]
x = range(len(resumen_canasta))
w = 0.35
ax.barh([i + w/2 for i in x], resumen_canasta['cob_cadenas'],  w, label='Cadenas promedio',  color='steelblue')
ax.barh([i - w/2 for i in x], resumen_canasta['cob_regiones'], w, label='Regiones promedio', color='seagreen')
ax.set_yticks(list(x))
ax.set_yticklabels(resumen_canasta['grupo_canasta'], fontsize=9)
ax.set_xlabel('Promedio de cadenas / regiones')
ax.axvline(MIN_CADENAS,  color='steelblue', linestyle=':', linewidth=1, alpha=0.7)
ax.axvline(MIN_REGIONES, color='seagreen',  linestyle=':', linewidth=1, alpha=0.7)
ax.set_title('Cobertura promedio por grupo')
ax.legend()

# Precio mediano por grupo
ax = axes[1]
resumen_sorted = resumen_canasta.sort_values('precio_mediano', ascending=True)
bars = ax.barh(resumen_sorted['grupo_canasta'], resumen_sorted['precio_mediano'],
               color=plt.cm.RdYlGn(resumen_sorted['precio_mediano'] / resumen_sorted['precio_mediano'].max()))
ax.set_xlabel('Precio mediano (pesos)')
ax.set_title('Precio mediano por grupo (en pesos)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_resumen_canasta.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Dispersión de precios por grupo (box plot) ----
fig, ax = plt.subplots(figsize=(14, 6))

orden = (
    df_canasta.groupby('grupo_canasta')['precio_mediano']
    .median().sort_values(ascending=False).index.tolist()
)

data_plot = [
    df_canasta[df_canasta['grupo_canasta'] == g]['precio_mediano'].dropna().values
    for g in orden
]

bp = ax.boxplot(data_plot, vert=False, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
colors = plt.cm.tab20.colors
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.75)

ax.set_yticks(range(1, len(orden) + 1))
ax.set_yticklabels(orden, fontsize=9)
ax.set_xlabel('Precio mediano por sucursal (pesos)', fontsize=10)
ax.set_title('Dispersión de precios por grupo de la canasta — Abril 2026', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_dispersion_precios.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exportación de resultados

In [ ]:
# Exportar la canasta a CSV
cols_export = [
    'grupo_canasta', 'id_producto', 'descripcion', 'marca',
    'presentacion', 'unidad', 'rubro', 'categoria', 'subcategoria',
    'n_cadenas', 'pct_cadenas', 'n_regiones', 'pct_regiones',
    'n_sucursales', 'pct_dias_promedio',
    'precio_mediano', 'precio_promedio', 'precio_p25', 'precio_p75',
    'score_cobertura'
]

canasta_export = (
    df_canasta[cols_export]
    .sort_values(['grupo_canasta', 'score_cobertura'], ascending=[True, False])
)

out_csv = f'{OUTPUT_DIR}/canasta_representativa_abril2026.csv'
canasta_export.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'Canasta exportada: {out_csv}')

# Exportar también la tabla de cobertura completa de candidatos
out_cob = f'{OUTPUT_DIR}/cobertura_todos_candidatos_abril2026.csv'
cand_con_maestro.sort_values('score_cobertura', ascending=False).to_csv(
    out_cob, index=False, encoding='utf-8-sig'
)
print(f'Cobertura completa exportada: {out_cob}')

# Resumen final
print()
print('=' * 70)
print('RESUMEN FINAL')
print('=' * 70)
print(f'Productos en la canasta: {len(canasta_export)}')
print(f'Candidatos totales (pasan filtros): {len(cand_con_maestro):,}')
print()
print(canasta_export.groupby('grupo_canasta').agg(
    productos       = ('id_producto',      'count'),
    cadenas_prom    = ('n_cadenas',         'mean'),
    regiones_prom   = ('n_regiones',        'mean'),
    precio_mediano  = ('precio_mediano',    'median')
).round(1).to_string())